# 10. requests로 데이터 가져오기

지금까지 데이터는 노트북 안에서 직접 만들었습니다. 이번에는 다른 서비스에 있는 데이터를 가져옵니다.

- 코드 칸 안의 `#>` 줄은 그 칸에서 할 일입니다
- 그 아래에 코드를 치고 Shift + Enter로 실행합니다

## 1. 밖에 있는 데이터

이번 주 로또 당첨번호는 노트북 안에 없습니다. 동행복권 서버에 있습니다.

아래 주소를 브라우저 새 탭에 붙여 넣어 열어 봅니다.

```
https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do
```

화면에 `{"resultCode":null, … "data":{"list":[{ … "ltEpsd":…` 같은 글이 보입니다. 08장에서 본 **JSON**입니다. 주소로 요청하면 서버가 JSON으로 답합니다. 이렇게 프로그램이 데이터를 주고받으라고 열어 둔 주소를 **API**라고 합니다.

## 2. `requests.get`

**기본 구조**

```
import requests

응답 = requests.get(주소, timeout=10)
응답.status_code              # 200이면 성공
응답.text                     # 받은 글 그대로
```

`requests`는 주소로 요청을 보내는 라이브러리입니다. 이 저장소에 설치되어 있습니다(05장 7절). `timeout=10`은 10초 안에 답이 없으면 기다리지 않고 멈추라는 뜻입니다.

In [ ]:
import requests

res = requests.get('https://www.naver.com')
# print(res.text)
# res.json()

In [ ]:
#> requests를 import하고, LOTTO_URL로 요청을 보내 r에 넣은 뒤 status_code를 확인합니다
LOTTO_URL = "https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do"
r = requests.get(LOTTO_URL)
print(r)

<Response [200]>


`200`은 "요청이 잘 처리됐다"는 뜻의 **상태 코드**입니다.

In [ ]:
#> r.text의 앞 150글자를 확인합니다
r.text[:150]

'{"resultCode":null,"resultMessage":null,"data":{"list":[{"winType0":0,"winType1":5,"winType2":3,"winType3":0,"gmSqNo":5133,"ltEpsd":1247,"tm1WnNo":8,"'

`r.text`는 문자열입니다. 브라우저에서 본 글과 같습니다.

## 3. `.json()`

**기본 구조**

```
데이터 = 응답.json()          # JSON 글 → 딕셔너리 · 리스트
```

`.json()`은 받은 JSON 글을 08장 `json.loads`처럼 딕셔너리로 바꿔 줍니다.

In [ ]:
#> r.json()을 data에 넣고, data의 종류와 키를 print 한 번으로 출력합니다
data = r.json()
print(data)

{'resultCode': None, 'resultMessage': None, 'data': {'list': [{'winType0': 0, 'winType1': 5, 'winType2': 3, 'winType3': 0, 'gmSqNo': 5133, 'ltEpsd': 1247, 'tm1WnNo': 8, 'tm2WnNo': 14, 'tm3WnNo': 21, 'tm4WnNo': 22, 'tm5WnNo': 42, 'tm6WnNo': 45, 'bnsWnNo': 24, 'ltRflYmd': '20261024', 'rnk1WnNope': 8, 'rnk1WnAmt': 3637439250, 'rnk1SumWnAmt': 29099514000, 'rnk2WnNope': 74, 'rnk2WnAmt': 65539446, 'rnk2SumWnAmt': 4849919004, 'rnk3WnNope': 2983, 'rnk3WnAmt': 1625853, 'rnk3SumWnAmt': 4849919499, 'rnk4WnNope': 154703, 'rnk4WnAmt': 50000, 'rnk4SumWnAmt': 7735150000, 'rnk5WnNope': 2591204, 'rnk5WnAmt': 5000, 'rnk5SumWnAmt': 12956020000, 'sumWnNope': 2748972, 'rlvtEpsdSumNtslAmt': 59490522503, 'wholEpsdSumNtslAmt': 118981044000, 'excelRnk': '1등'}]}}


당첨 정보는 `data` 안의 `"data"` 안의 `"list"` 첫 번째에 있습니다.

In [ ]:
#> data["data"]["list"][0]을 item에 넣고 확인합니다
item = data['data']['list'][0]
# print(item)

키가 많습니다. 쓸 것은 이 넷입니다.

| 키 | 뜻 |
| --- | --- |
| `ltEpsd` | 회차 |
| `ltRflYmd` | 추첨일(`"20261024"` 모양 문자열) |
| `tm1WnNo` ~ `tm6WnNo` | 당첨번호 여섯 개 |
| `bnsWnNo` | 보너스 번호 |

In [ ]:
#> item에서 회차와 추첨일을 print 한 번으로 출력합니다
print(item['ltEpsd'], item['ltRflYmd'])

1247 20261024


In [ ]:
#> tm1WnNo부터 tm6WnNo까지를 for로 하나씩 꺼내 lucky에 넣고, bnsWnNo를 bonus에 넣어 확인합니다
lucky = []
for i in range(1, 7):
    number = item[f'tm{i}WnNo']
    lucky.append(number)
bonus = item['bnsWnNo']
print(lucky, bonus)
    

[8, 14, 21, 22, 42, 45] 24


키 이름을 f-string으로 만들어 여섯 번 꺼냈습니다(01장 f-string). 주석은 같은 일을 03장 컴프리헨션으로 줄인 모양입니다.

## 4. `params`

**기본 구조**

```
응답 = requests.get(주소, params={"이름": 값}, timeout=10)
응답.url                      # 실제로 요청한 주소
```

지난 회차를 보려면 주소 뒤에 `?srchLtEpsd=회차`를 붙입니다. 직접 붙이지 않고 `params`에 딕셔너리로 넣으면 `requests`가 붙여 줍니다.

In [ ]:
#> params로 1100회를 요청해 r1100에 넣고, r1100.url을 확인합니다
r1100 = requests.get(LOTTO_URL, params={'srchLtEpsd': 1100})
print(r1100)

<Response [200]>


In [ ]:
#> r1100에서 1100회 당첨번호 여섯 개를 numbers1100에 꺼내 확인합니다
data = r1100.json()
item = data['data']['list'][0]
numbers1100 = []

for i in range(1, 7):
    numbers1100.append(item[f'tm{i}WnNo'])

numbers1100

[17, 26, 29, 30, 31, 43]

회차 여러 개를 한 번에 받을 수도 있습니다. 시작 회차 `srchStrLtEpsd`와 끝 회차 `srchEndLtEpsd` 두 값을 넣습니다.

In [ ]:
#> 3절에서 본 최근 회차 번호를 latest에 직접 넣고, latest - 9부터 latest까지 params 두 값으로 요청해 r_recent에 넣고, r_recent.url을 확인합니다
latest = 1247

r_recent = requests.get(LOTTO_URL, params={'srchStrLtEpsd': latest - 99, 'srchEndLtEpsd': latest})
r_recent.url

'https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do?srchStrLtEpsd=1148&srchEndLtEpsd=1247'

값이 둘이면 `requests`가 `&`로 이어 붙입니다.

In [ ]:
#> r_recent의 list를 recent에 넣고, 회차들을 episodes에 모아 몇 개인지와 함께 print 한 번으로 출력합니다
recent = r_recent.json()['data']['list']
print(len(recent))

100


`list`에 회차 10개가 최신 회차부터 들어 있습니다. 그래서 `list`가 리스트였습니다.

## 5. 로또 5,000원

1. 로또 5,000원어치(5세트)를 삽니다. 1부터 45까지 가운데 겹치지 않게 6개씩(05장 `random.sample`), 작은 순서로
2. 최신 회차 당첨번호와 보너스를 받습니다(3절)
3. 등수를 돌려주는 함수 `grade(my, lucky, bonus)`를 만들고(04장), 세트마다 등수를 출력합니다. 1등 · 3등 · 4등 · 5등 · 꽝을 먼저 만들고, 2등(5개 + 보너스)은 마지막에 넣습니다

| 등수 | 조건 |
| --- | --- |
| 1등 | 6개 |
| 2등 | 5개 + 보너스 |
| 3등 | 5개 |
| 4등 | 4개 |
| 5등 | 3개 |

In [ ]:
#> 1. random을 import하고, 5세트를 my_sets에 담아 출력합니다
import random
my_sets = []

for i in range(5):
    my_number = random.sample(range(1, 46), 6)
    my_sets.append(sorted(my_number))

print(my_sets)
# 1등 확인용 데이터
my_sets = [[8, 14, 21, 22, 42, 45], [1, 6, 17, 25, 40, 43], [8, 13, 37, 38, 39, 45], [4, 15, 28, 33, 42, 44], [7, 11, 27, 36, 39, 42]]

[[2, 3, 9, 16, 18, 45], [7, 8, 10, 23, 28, 34], [21, 27, 31, 37, 43, 45], [1, 8, 10, 21, 33, 45], [2, 4, 5, 32, 39, 44]]


In [ ]:
#> 2. 최신 회차의 당첨번호를 lucky에, 보너스를 bonus에 넣어 회차와 함께 출력합니다
res = requests.get(LOTTO_URL)
data = res.json()['data']['list'][0]

lucky = []
for i in range(1, 7):
    lucky.append(data[f'tm{i}WnNo'])
bonus = data['bnsWnNo']
print(lucky, bonus, data['ltEpsd'])


[8, 14, 21, 22, 42, 45] 24 1247


In [ ]:
#> 3. 맞힌 개수(set의 &)로 등수를 돌려주는 grade(my, lucky, bonus)를 만들고, 세트마다 번호와 등수를 출력합니다

def grade(my, lucky, bonus):
    count = len(set(my) & set(lucky))

    if count == 6:
        return '1등'
    elif count == 5 and bonus in my:
        return '2등'
    elif count == 5:
        return '3등' 
    elif count == 4:
        return '4등'
    elif count == 3:
        return '5등'
    else:
        return '꽝'
    

# grade([1, 2, 3], [1, 2, 3, 4, 5, 6], 7)

for my_set in my_sets:
    print(grade(my_set, lucky, bonus))


1등
꽝
꽝
꽝
꽝


더 해 볼 것: 1등이 나올 때까지 몇 세트를 사야 하는지 `while`로 세기 · 지난 회차로도 해 보기(`params`)

## 6. 최근 100회로 질문하기

범위로 최근 100회를 받아, 데이터로 질문에 답합니다.

| 키 | 뜻 |
| --- | --- |
| `rnk1WnAmt` | 1등 한 사람이 받은 당첨금(원) |
| `rnk1WnNope` | 1등 당첨자 수(명) |

번호 키(`tm1WnNo` ~ `tm6WnNo` · `bnsWnNo`)는 3절 표에 있습니다.

In [ ]:
#> 1. 최근 회차번호를 latest에 넣고, latest - 99부터 latest까지 범위를 요청해 draws에 넣고, 몇 회차인지 확인합니다
latest = 1247
res = requests.get(LOTTO_URL, params={'srchStrLtEpsd': latest-99, 'srchEndLtEpsd': latest})

draws = res.json()['data']['list']

**질문 1. 가장 많이 나온 번호는?**

In [ ]:
#> 2. 번호마다 몇 번 나왔는지 counts 딕셔너리에 셉니다(03장 세기)
counts = {}
# 모든 회차를 반복
for draw in draws:
    # 회차안에서 6번 반복
    for i in range(1, 7):
        number = draw[f'tm{i}WnNo']

        # 이미 추가한 경우
        if number in counts:
            counts[number] += 1
        else:
            counts[number] = 1

print(counts)

{8: 15, 14: 8, 21: 11, 22: 13, 42: 15, 45: 12, 16: 18, 23: 14, 24: 16, 27: 20, 28: 17, 4: 12, 5: 10, 25: 14, 32: 14, 1: 13, 13: 19, 18: 15, 26: 11, 34: 10, 38: 19, 9: 16, 43: 9, 44: 12, 2: 11, 10: 10, 31: 16, 41: 11, 7: 14, 11: 10, 19: 13, 20: 13, 33: 10, 36: 13, 37: 14, 40: 11, 12: 12, 29: 13, 6: 13, 15: 17, 39: 15, 35: 13, 3: 16, 30: 12, 17: 10}


In [ ]:
#> 3. counts에서 가장 많이 나온 번호와 횟수, 가장 적게 나온 번호와 횟수를 찾아 출력합니다
most = 0
least = 100
most_number = 8
least_number = 8

for n in counts.keys():
    # most값보다 더 큰 반복횟수가 등장하면 원래 데이터 수정
    if counts[n] > most:
        most_number = n
        most = counts[n]
        
    if counts[n] < least:
        least_number = n
        least = counts[n]

print(most_number, most)
print(least_number, least)
    


27 20
14 8


**질문 2. 1등 당첨금이 가장 컸던 회차와 가장 작았던 회차는?**

In [ ]:
#> 4. draws에서 rnk1WnAmt가 가장 큰 회차와 가장 작은 회차를 찾아, 회차 · 당첨금 · 당첨자 수를 출력합니다
result = []
for draw in draws:
    amount = draw['rnk1WnAmt']
    count = draw['rnk1WnNope']
    epsd = draw['ltEpsd']
    result.append((amount, count, epsd))

sorted_result = sorted(result)
sorted_result[0]
sorted_result[-1]
print(sorted_result[0], sorted_result[-1])
    

(823931021, 36, 1162) (5001713625, 6, 1208)


**질문 3. 내가 고른 번호로 100회를 다 샀다면?**

In [ ]:
#> 5. 내가 고른 번호 6개를 my에 넣고, 회차마다 몇 개 맞혔는지 세어 맞힌 개수마다 몇 번인지 matches에 셉니다. 쓴 돈도 출력합니다
my = [1, 2, 3, 4, 5, 6]
matches = {}
for draw in draws:
    numbers = []
    for i in range(1, 7):
        numbers.append(draw[f'tm{i}WnNo'])
    count = len(set(my) & set(numbers))

    if count in matches:
        matches[count] += 1
    else:
        matches[count] = 1

print(matches)

    


{0: 44, 2: 15, 1: 39, 3: 2}


3개를 맞히면 5등(5,000원), 4개면 4등(50,000원)입니다.

## 정리

| 할 일 | 방법 |
| --- | --- |
| 요청 보내기 | `requests.get(주소, timeout=10)` |
| 성공했나 | `응답.status_code` — `200` |
| 받은 글 · JSON | `응답.text` · `응답.json()` |
| 주소 뒤 값 | `params={"이름": 값}` · 실제 주소는 `응답.url` |
| 값이 여럿 | `params={"이름1": 값1, "이름2": 값2}` — `&`로 이어 붙는다 |